In [11]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [12]:
## Load the trained model,scaler pickle ,onehot
model=load_model('model.h5')

## load the encoder and scaler
with open('onehot_encoder_geo.pkl','rb') as file:
    label_encoder_geo=pickle.load(file)

with open('label_encoder_gender.pkl','rb') as file:
    label_encoder_gender = pickle.load(file)

with open('scaler.pkl','rb') as file:
    scaler = pickle.load(file)
    

In [13]:
# Example input data
input_data = {
    'CreditScore':600,
    'Geography':'France',
    'Gender':'Male',
    'Age':40,
    'Tenure':3,
    'Balance':60000,
    'NumberOfProducts':2,
    'HasCrCard':1,
    'isActiveMember':1,
    'EstimatedSalary':50000
}

In [14]:
## one-hot encode 'Geography'
geo_encoded = label_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded,columns=label_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

c:\AI\Projects\End to End Deep Learning Project - ANN\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [15]:
input_df=pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumberOfProducts,HasCrCard,isActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [16]:
input_df['Gender']=label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumberOfProducts,HasCrCard,isActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [25]:
## concatination one hot encoded
input_df=pd.concat([input_df.drop("Geography",axis=1),geo_encoded_df],axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumberOfProducts,HasCrCard,isActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [26]:
##Scaling the input dada
input_scaled=scaler.transform(input_df)
input_scaled

ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- NumberOfProducts
- isActiveMember
Feature names seen at fit time, yet now missing:
- IsActiveMember
- NumOfProducts
- RowNumber


In [27]:
# Step 1: Convert to DataFrame
input_df = pd.DataFrame([input_data])

# Step 2: Encode Gender
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])

# Step 3: One-hot encode Geography
geo_encoded = label_encoder_geo.transform(
    pd.DataFrame(input_df[['Geography']])
).toarray()

geo_df = pd.DataFrame(
    geo_encoded,
    columns=label_encoder_geo.get_feature_names_out(['Geography'])
)

# Step 4: Combine and drop original column
final_input = pd.concat([
    input_df.drop('Geography', axis=1),
    geo_df
], axis=1)

# Step 5: Align columns with training
final_input = final_input.reindex(columns=scaler.feature_names_in_, fill_value=0)

# Step 6: Scale
input_scaled = scaler.transform(final_input)

input_scaled

array([[-1.73595179, -0.53598516,  0.91324755,  0.10479359, -0.69539349,
        -0.25781119, -2.6418115 ,  0.64920267, -1.02583358, -0.87683221,
         1.00150113, -0.57946723, -0.57638802]])

In [28]:
## Prediction 
prediction = model.predict(input_scaled)
prediction


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step


array([[0.7992821]], dtype=float32)

In [29]:
prediction_proba=prediction[0][0]

In [30]:
prediction_proba

0.7992821

In [31]:
if prediction_proba>0.5:
    print('The customer is likely to churn.')
else:
    print('The customer is not likely to churn.')

The customer is likely to churn.
